In [1]:
from typing import List, Dict, Any
import csv
import os
import weaviate
from weaviate.classes.config import Configure, Property, DataType, Tokenization
from weaviate.util import generate_uuid5
from weaviate.classes.init import AdditionalConfig, Timeout
import tqdm
from weaviate.classes.query import Filter
import flask_app
from utils import ( 
    suppress_subprocess_output,
    kill_processes_on_ports,
    generate_with_single_input
)

/Users/elahe/venvs/jlab/lib/python3.13/site-packages/authlib/integrations/httpx_client/assertion_client.py:5: AuthlibDeprecationWarning: The httpx module is deprecated; please use httpx2 instead.
  from ._compat import httpx2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

 * Serving Flask app 'flask_app'
 * Debug mode: off


In [2]:
def get_chunks_from_csv(file_path: str) -> List[Dict[str, Any]]:
    chunks = []

    with open(file_path, "r", encoding="utf-8") as f:
        source_csv = csv.DictReader(f)

        for row in source_csv:
            chunk_text = " | ".join(
                f"{column}: {row[column]}"
                for column in source_csv.fieldnames
                if row[column]  # نادیده گرفتن ستون‌های خالی
            )
            
            chunk_data = {
                "chunk": chunk_text,
                **row
            }
            chunks.append(chunk_data)
    return chunks

In [3]:
created_chuncks = get_chunks_from_csv("faq_qa_pairs.csv")

In [4]:
import os
import weaviate

collection_base = os.getenv('COLLECTION_M3', './')
persistence_path = os.path.abspath(os.path.join(collection_base, 'chatbot'))

client = weaviate.connect_to_embedded(
    port=8079,
    grpc_port=50050,
    persistence_data_path=persistence_path,
    environment_variables={
        "ENABLE_API_BASED_MODULES": "true",
        "ENABLE_MODULES": "text2vec-transformers",
        "TRANSFORMERS_INFERENCE_API": "http://127.0.0.1:5050",
    }
)


{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"warning","log_level_env":"","msg":"log level not recognized, defaulting to info","time":"2026-10-06T19:06:11+04:00"}
{"action":"startup","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"Feature flag LD integration disabled: could not locate WEAVIATE_LD_API_KEY env variable","time":"2026-10-06T19:06:11+04:00"}
{"action":"startup","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","default_vectorizer_module":"none","level":"info","msg":"the default vectorizer modules is set to \"none\", as a result all new schema classes without an explicit vectorizer setting, will use this vectorizer","time":"2026-10-06T19:06:11+04:00"}
{"action":"startup","auto_schema_enabled":{},"build_git_commit":"62dcafac32","build_go_version":

In [5]:
# Creating the collection
if not client.collections.exists("chunking_example"):
    collection = client.collections.create(
            name='chunking_example',

            vectorizer_config=[Configure.NamedVectors.text2vec_transformers(
                    name="vector", # This is the name you will need to access the vectors of the objects in your collection
                    #source_properties=['chunk'], # which properties should be used to generate a vector, they will be appended to each other when vectorizing
                    vectorize_collection_name = False, # This tells the client to not vectorize the collection name. 
                                                       # If True, it will be appended at the beginning of the text to be vectorized
                    inference_url="http://127.0.0.1:5050", # Since we are using an API based vectorizer, you need to pass the URL used to make the calls 
                                                           # This was setup in our Flask application
                )],

            properties=[  # Define properties
            Property(name="chunk", data_type=DataType.TEXT),
            Property(name="question", data_type=DataType.TEXT),
            Property(name="answer", data_type=DataType.TEXT),
            Property(name="category", data_type=DataType.TEXT),

        ]
        )
else:
    collection = client.collections.get("chunking_example")

In [6]:
if collection.aggregate.over_all().total_count == 0:
    with collection.batch.fixed_size(
        batch_size=1,
        concurrent_requests=20
    ) as batch:

        for chunk_obj in tqdm.tqdm(created_chuncks):
            batch.add_object(
                properties=chunk_obj,
                uuid=generate_uuid5(chunk_obj)
            )

print(f"Total count: {collection.aggregate.over_all().total_count}")

Total count: 134


In [7]:
user_question = "شرایط دریافت ضمانت‌نامه چیست؟"

response = collection.query.near_text(
    user_question,
    limit=3
)

for i, obj in enumerate(response.objects):
    print(f"===== Result {i+1} =====")
    print(obj.properties["chunk"])
    print()

===== Result 1 =====
category: صکوک | question: سقف میزان ضمانت نامه صادره برای شرکت‌های دانش‌بنیان بر چه اساسی تعیین می‌شود؟ | answer: میزان ضمانت نامه قابل ارائه به شرکت‌های دانش‌بنیان متناسب با برنامه شرکت برای هزینه های سرمایه ثابت و سرمایه در گردش قابل محاسبه می باشد در این راستا طرح شرکت به لحاظ مسائل فنی اقتصادی و بازار بررسی شده و هزینه های مورد نیاز توسعه زیرساخت و یا هزینه های جاری تولید محصول دانش بنیان محاسبه می شود و سقف تامین مالی مورد نیاز مشخص و متناسب با رفتار مالی و اعتباری شرکت درصد پذیرش ریسک صندوق محاسبه و میزان سقف ضمانت نامه قابل اعطا محاسبه می شود. ارزیابی تقاضاها بر مبنای اطلاعات ارائه شده توسط شرکت انجام می‌شود و هر چه اطلاعات ارائه شده دقیق­تر و کامل­تر باشد به تصمیم گیری بهتر و سریعتر کمک می کند. از این رو تکمیل دقیق کاربرگ های تقاضا و همکاری در فرایند ارزیابی از اهمیت زیادی برخوردار است.

===== Result 2 =====
category: ضمانت‌نامه | question: شرکت دانش بنیان برای دریافت ضمانت نامه باید واجد چه شرایطی باشد؟ | answer: دانش بنیان بودن عنوان قرارداد نداشتن اقساط

In [8]:
PROMPT = """You are a helpful assistant.
Answer the user's question using only the information provided in the context.
If the answer is not in the context, say that you don't know.

User question:
{question}

Context:
{context}

Answer:
"""

user_question = "شرایط دریافت ضمانت‌نامه چیست؟"
response = collection.query.near_text(user_question,limit=3)
context_string = ""

for obj in response.objects:
    context_string += obj.properties["chunk"] + "\n"

prompt = PROMPT.format(question=user_question,context=context_string)

# Send prompt to the LLM
response = generate_with_single_input(prompt,role="user")

print("Answer:")
print(response["content"])

Answer:
شرایط دریافت ضمانت‌نامه عبارتند از:
۱. بودن در عنوان «دانش‌بنیان» (در قرارداد).
۲. نداشتن اقساط معوق و چک برگشتی.
۳. عدم شمول ماده ۱۴۱ قانون تجارت.

علاوه بر موارد فوق، شرایط صدور ضمانت‌نامه با توجه به اعتبار شرکت مشخص می‌شود.


In [ ]:
client.close()